In [5]:

!pip install ultralytics
!pip install tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.1/977.1 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 55.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [1]:

from torchvision.datasets import VOCDetection
import os

# Download Pascal VOC 2012
dataset = VOCDetection(root="voc_data", year="2012", image_set="train", download=True)
print("VOC 2012 dataset downloaded.")


100%|██████████| 2.00G/2.00G [00:50<00:00, 39.7MB/s]


VOC 2012 dataset downloaded.


In [19]:

import os
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm

VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]
class_to_id = {cls: idx for idx, cls in enumerate(VOC_CLASSES)}

image_dir = "custom_voc/images/train"
label_dir = "custom_voc/labels/train"
os.makedirs(image_dir, exist_ok=True)
os.makedirs(label_dir, exist_ok=True)

voc_root = "voc_data/VOCdevkit/VOC2012"

image_list = os.listdir(os.path.join(voc_root, "JPEGImages"))[:20]

for img_file in tqdm(image_list):
    base = os.path.splitext(img_file)[0]
    xml_path = os.path.join(voc_root, "Annotations", base + ".xml")
    img_path = os.path.join(voc_root, "JPEGImages", img_file)
    new_img_path = os.path.join(image_dir, img_file)
    new_label_path = os.path.join(label_dir, base + ".txt")

    if not os.path.exists(xml_path): continue

    tree = ET.parse(xml_path)
    root = tree.getroot()
    img = Image.open(img_path)
    w, h = img.size
    img.save(new_img_path)

    with open(new_label_path, "w") as f:
        for obj in root.findall("object"):
            cls = obj.find("name").text
            if cls not in class_to_id:
                continue
            label_id = class_to_id[cls]
            bbox = obj.find("bndbox")
            x1 = int(float(bbox.find("xmin").text))
            y1 = int(float(bbox.find("ymin").text))
            x2 = int(float(bbox.find("xmax").text))
            y2 = int(float(bbox.find("ymax").text))
            x_center = (x1 + x2) / 2 / w
            y_center = (y1 + y2) / 2 / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h
            f.write(f"{label_id} {x_center} {y_center} {bw} {bh}\n")


100%|██████████| 20/20 [00:00<00:00, 201.72it/s]


In [34]:
import os

full_path = os.path.abspath("custom_voc")

yaml_text = f"""\
path: {full_path}
train: images/train
val: images/train

nc: 20
names: ['aeroplane','bicycle','bird','boat','bottle','bus','car','cat','chair',
        'cow','diningtable','dog','horse','motorbike','person','pottedplant',
        'sheep','sofa','train','tvmonitor']
"""

with open("custom_voc.yaml", "w") as f:
    f.write(yaml_text)

print("✅ custom_voc.yaml written with absolute path:")
print(full_path)


✅ custom_voc.yaml written with absolute path:
/content/custom_voc


In [46]:
!yolo task=detect mode=train model=yolov8n.pt data=custom_voc.yaml epochs=5 imgsz=416

Ultralytics 8.3.100 🚀 Python-3.11.11 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=custom_voc.yaml, epochs=5, time=None, patience=100, batch=16, imgsz=416, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train9, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, show_boxes=True, line_w

In [47]:
for root, dirs, files in os.walk("runs/detect"):
    for name in files:
        if name.endswith("best.pt"):
            print("✅ Found best.pt:", os.path.join(root, name))

✅ Found best.pt: runs/detect/train7/weights/best.pt
✅ Found best.pt: runs/detect/train9/weights/best.pt
✅ Found best.pt: runs/detect/train8/weights/best.pt
✅ Found best.pt: runs/detect/train5/weights/best.pt


In [48]:
!yolo task=detect mode=val model=runs/detect/train9/weights/best.pt data=custom_voc.yaml

Ultralytics 8.3.100 🚀 Python-3.11.11 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 72 layers, 3,009,548 parameters, 0 gradients, 8.1 GFLOPs
val: Scanning /content/custom_voc/labels/train.cache... 200 images, 0 backgrounds, 0 corrupt: 100% 200/200 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% 13/13 [00:23<00:00,  1.82s/it]
                   all        200        431      0.126      0.364      0.168      0.152
             aeroplane         10         11     0.0748      0.727      0.176      0.159
               bicycle         10         17          0          0          0          0
                  bird         13         15     0.0526      0.933      0.162      0.151
                  boat         10         16     0.0667       0.25      0.224      0.174
                bottle          8         10    0.00741        0.1    0.00702    0.00632
                   bus          5          6          1 

In [49]:
!yolo task=detect mode=predict model=runs/detect/train9/weights/best.pt source=custom_voc/images/train conf=0.1 save=True

Ultralytics 8.3.100 🚀 Python-3.11.11 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
Model summary (fused): 72 layers, 3,009,548 parameters, 0 gradients, 8.1 GFLOPs

image 1/200 /content/custom_voc/images/train/2007_000549.jpg: 416x320 (no detections), 97.6ms
image 2/200 /content/custom_voc/images/train/2007_002046.jpg: 416x384 (no detections), 109.2ms
image 3/200 /content/custom_voc/images/train/2007_003134.jpg: 320x416 (no detections), 89.7ms
image 4/200 /content/custom_voc/images/train/2007_003503.jpg: 288x416 (no detections), 86.3ms
image 5/200 /content/custom_voc/images/train/2007_005227.jpg: 288x416 (no detections), 80.1ms
image 6/200 /content/custom_voc/images/train/2007_005759.jpg: 320x416 (no detections), 85.8ms
image 7/200 /content/custom_voc/images/train/2007_005989.jpg: 320x416 (no detections), 81.8ms
image 8/200 /content/custom_voc/images/train/2007_006066.jpg: 288x416 (no detections), 74.3ms
image 9/200 /content/custom_voc/images/train/2007_006232.jpg: 288x416 (no detections),

In [45]:
sample_labels = os.listdir("custom_voc/labels/train")[:5]
for file in sample_labels:
    with open(os.path.join("custom_voc/labels/train", file)) as f:
        print(f"{file}:", f.readlines())


2008_002794.txt: ['14 0.878 0.7026666666666667 0.116 0.44533333333333336\n', '14 0.513 0.6973333333333334 0.094 0.47733333333333333\n', '14 0.11 0.6933333333333334 0.216 0.496\n', '14 0.498 0.4746666666666667 0.124 0.208\n']
2010_001515.txt: ['18 0.456 0.554 0.9066666666666666 0.812\n']
2011_006485.txt: ['14 0.4743975903614458 0.359 0.29819277108433734 0.23\n']
2008_006050.txt: ['7 0.481 0.5016778523489933 0.958 0.8691275167785235\n']
2008_006121.txt: ['3 0.43333333333333335 0.428 0.47733333333333333 0.176\n', '14 0.6813333333333333 0.509 0.104 0.182\n']
